# 02 Model Training and Evaluation

## Prediction question
For a **reported strike with a known damage label**, classify whether aircraft damage was reported. This is conditional on a strike report; it is not a model of whether an ordinary flight will experience a strike.

We compare a majority-class baseline, balanced Logistic Regression, and a modest Random Forest. We choose between the two learned models using validation average precision because the damage class may be less common. The 2025 test set remains untouched until the final evaluation.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             classification_report)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = Path.cwd()
INPUT = PROJECT_DIR / "data" / "processed" / "strike_model_table.csv"
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT.exists():
    raise FileNotFoundError("Run 01_PREPROCESSING_AND_EDA.ipynb first.")
data = pd.read_csv(INPUT, low_memory=False)

categorical_features = ["INCIDENT_MONTH", "TIME_OF_DAY", "PHASE_OF_FLT", "AC_CLASS", "AC_MASS", "SIZE", "SPECIES"]
numeric_features = ["BIRDS_STRUCK_LOG1P", "HEIGHT_LOG1P"]
feature_columns = categorical_features + numeric_features
target_column = "AIRCRAFT_DAMAGED"

In [ ]:
# Chronological split: training through 2022, validation 2023-24, final holdout 2025.
train = data.loc[data["INCIDENT_YEAR"] <= 2022].copy()
validation = data.loc[data["INCIDENT_YEAR"].between(2023, 2024)].copy()
test = data.loc[data["INCIDENT_YEAR"].eq(2025)].copy()
if min(len(train), len(validation), len(test)) == 0:
    raise ValueError("One chronological split is empty. Check the source years and adjust the documented cutoffs.")

def describe_split(name, frame):
    return {"split":name,"rows":len(frame),"first_year":int(frame.INCIDENT_YEAR.min()),
            "last_year":int(frame.INCIDENT_YEAR.max()),"damage_rate":float(frame[target_column].mean())}

split_summary = pd.DataFrame([describe_split(n,d) for n,d in
                              [("train",train),("validation",validation),("test",test)]])
display(split_summary)
X_train, y_train = train[feature_columns], train[target_column]
X_val, y_val = validation[feature_columns], validation[target_column]
X_test, y_test = test[feature_columns], test[target_column]

## Why the pipeline matters
Each model gets its own preprocessing pipeline. Missing-value fill values, one-hot categories, and (for logistic regression) scaling are learned from the training split only. Categories not seen during training are ignored safely at validation/test time. Rare species categories are grouped by the encoder’s minimum-frequency rule.

In [ ]:
def make_preprocessor(scale_numeric):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipe = Pipeline(numeric_steps)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=25)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_features),
        ("categorical", categorical_pipe, categorical_features),
    ], remainder="drop")

models = {
    "Balanced Logistic Regression": Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=True)),
        ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("classifier", RandomForestClassifier(
            n_estimators=200, min_samples_leaf=10, class_weight="balanced_subsample",
            random_state=42, n_jobs=-1
        )),
    ]),
}

def score_model(name, model, X, y):
    probability = model.predict_proba(X)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    return {
        "model":name,
        "accuracy":accuracy_score(y,prediction),
        "precision":precision_score(y,prediction,zero_division=0),
        "recall":recall_score(y,prediction,zero_division=0),
        "f1":f1_score(y,prediction,zero_division=0),
        "roc_auc":roc_auc_score(y,probability),
        "average_precision":average_precision_score(y,probability),
    }, probability, prediction

In [ ]:
comparison_rows=[]
validation_scores={}
for name, model in models.items():
    model.fit(X_train, y_train)
    row, probability, prediction = score_model(name, model, X_val, y_val)
    comparison_rows.append(row)
    validation_scores[name]=(probability,prediction)

# The majority classifier provides an accuracy reference; it will have no recall
# when damage is the minority class, which is why accuracy is not the selection metric.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_val)
baseline_row={"model":"Majority baseline", "accuracy":accuracy_score(y_val,base_pred),
              "precision":precision_score(y_val,base_pred,zero_division=0),
              "recall":recall_score(y_val,base_pred,zero_division=0),
              "f1":f1_score(y_val,base_pred,zero_division=0),
              "roc_auc":np.nan,"average_precision":average_precision_score(y_val,baseline.predict_proba(X_val)[:,1])}
comparison_rows.insert(0,baseline_row)
validation_results=pd.DataFrame(comparison_rows).sort_values("average_precision",ascending=False)
display(validation_results)

In [ ]:
selected_name = validation_results.loc[
    validation_results["model"].ne("Majority baseline")
].iloc[0]["model"]
print("Selected on validation average precision:", selected_name)

# Refit the selected pipeline on train + validation, then evaluate 2025 once.
train_validation=pd.concat([train,validation],ignore_index=True)
X_train_validation=train_validation[feature_columns]
y_train_validation=train_validation[target_column]
final_model=models[selected_name]
final_model.fit(X_train_validation,y_train_validation)
test_probability=final_model.predict_proba(X_test)[:,1]
test_prediction=(test_probability>=0.5).astype(int)

test_metrics={
    "model":selected_name,"test_year":2025,"test_rows":len(test),
    "damage_rate":float(y_test.mean()),
    "accuracy":accuracy_score(y_test,test_prediction),
    "precision":precision_score(y_test,test_prediction,zero_division=0),
    "recall":recall_score(y_test,test_prediction,zero_division=0),
    "f1":f1_score(y_test,test_prediction,zero_division=0),
    "roc_auc":roc_auc_score(y_test,test_probability),
    "average_precision":average_precision_score(y_test,test_probability),
}
display(pd.DataFrame([test_metrics]))
print(classification_report(y_test,test_prediction,target_names=["No reported damage","Damage reported"],zero_division=0))
cm=confusion_matrix(y_test,test_prediction,labels=[0,1])
display(pd.DataFrame(cm,index=["Actual no damage","Actual damage"],columns=["Predicted no damage","Predicted damage"]))

In [ ]:
# Report feature counts before and after one-hot encoding for the project rubric.
preprocessor=final_model.named_steps["preprocess"]
encoded_feature_names=preprocessor.get_feature_names_out()
print("Source fields in raw export:", len(pd.read_csv(PROJECT_DIR / "data" / "interim" / "strike_reports_standardized.csv", nrows=1).columns))
print("Selected source predictors before encoding:", len(feature_columns))
print("Encoded model features after preprocessing:", len(encoded_feature_names))

classifier=final_model.named_steps["classifier"]
if hasattr(classifier,"feature_importances_"):
    importance=pd.Series(classifier.feature_importances_,index=encoded_feature_names).sort_values(ascending=False).head(15)
elif hasattr(classifier,"coef_"):
    importance=pd.Series(np.abs(classifier.coef_[0]),index=encoded_feature_names).sort_values(ascending=False).head(15)
else:
    importance=pd.Series(dtype=float)
display(importance.rename("importance").to_frame())

In [ ]:
OUTPUT_DIR=PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / "test_metrics.csv",index=False)
predictions=pd.DataFrame({
    "actual_damage":y_test.to_numpy(),
    "predicted_damage":test_prediction,
    "damage_score":test_probability,
    "incident_year":test["INCIDENT_YEAR"].to_numpy(),
})
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv",index=False)
joblib.dump(final_model,OUTPUT_DIR / "models" / "selected_damage_model.joblib")
(OUTPUT_DIR / "selected_model.json").write_text(json.dumps({"selected_model":selected_name,"threshold":0.5,"selection_metric":"validation average precision"},indent=2),encoding="utf-8")
print("Saved test_metrics.csv, test_predictions.csv, selected model, and selection notes.")

## Metric interpretation checklist
- **Precision:** of reports flagged as damaging, how many had a damage label?
- **Recall:** of reports with damage, how many did the model catch?
- **F1:** balance of precision and recall at the fixed 0.5 cutoff.
- **Average precision / PR-AUC:** primary model-selection metric for a potentially imbalanced damage label.
- **ROC-AUC:** ranking quality across thresholds; it is not a probability-calibration score.
- **Confusion matrix:** counts false alarms and missed damage reports.

Do not promise a target accuracy such as 90%. The honest result depends on the held-out year and reporting patterns. The class proportions and sample counts must accompany the headline scores.